# ResNet-50 CNN baseline and multi-generator evaluation

This notebook trains an ImageNet-pretrained ResNet-50 to classify **real (0)** versus **AI-generated (1)** images. It supports:

- single- or multi-generator training;
- a fixed training budget across generator combinations;
- reproducible training with checkpointing and early stopping;
- seen- and unseen-generator evaluation;
- accuracy, F1-score, AUROC, confusion matrices, and prediction exports;
- consolidated CSV/JSON results for the final report.

## Expected GenImage Compact layout

Point `data_root` at the `GenImage_Compact` directory. Manifests are optional: by default, the
notebook scans the actual image folders and infers metadata from paths such as:

```text
ADM/train/ai/00002.png      -> generator=ADM, split=train, label=ai
ADM/val/nature/00347.jpeg   -> generator=ADM, split=val, label=nature
```

This works even when a generator has no JSON manifest or has a JSON file containing only filenames.
If your team later creates one combined manifest with `path`, `label`, `generator`, and `split`, set
`manifest_json` to it to skip the folder scan.

The official `val` split is reserved for final evaluation. To avoid tuning on reported results, the notebook makes a small, deterministic internal validation subset from the selected generators' official `train` rows.

In [ ]:
# Run only if the environment is missing packages.
%pip install -q torch torchvision pandas numpy scikit-learn matplotlib seaborn pillow tqdm pyyaml

In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import os
import random
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Optional, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from PIL import Image, ImageFile
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ResNet50_Weights, resnet50
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Configuration

Edit this cell before running. `TRAIN_GENERATORS` controls whether the run is a single- or multi-generator experiment.

In [ ]:
@dataclass
class Config:
    # Data
    data_root: str = "/project2/yzhao010_1531/csci566-genimage/data/extracted/GenImage_Compact"
    # Optional *combined* JSON manifest. Leave as None to discover images from folders.
    manifest_json: Optional[str] = None
    train_generators: list[str] = field(default_factory=lambda: ["stable_diffusion_v_1_4"])
    run_name: str = "resnet50_k1_sdv14_seed0"
    output_root: str = "./cnn_outputs"

    # Fair-comparison budgets. Set to None to use every available row.
    # First run with None; then set identical totals supported by every planned combination.
    train_fake_budget: Optional[int] = None
    train_real_budget: Optional[int] = None
    val_fake_budget: Optional[int] = None
    val_real_budget: Optional[int] = None

    # Model/training
    image_size: int = 224
    batch_size: int = 64
    num_workers: int = 4
    epochs: int = 15
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    early_stopping_patience: int = 4
    seed: int = 0
    pretrained: bool = True
    use_amp: bool = True
    decision_threshold: float = 0.5
    optimize_threshold_on_validation: bool = False
    internal_val_fraction: float = 0.10

    # Debugging: set a small value (e.g. 200) for a fast end-to-end smoke test.
    debug_max_rows_per_split: Optional[int] = None


CFG = Config()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_DIR = Path(CFG.output_root) / CFG.run_name
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(asdict(CFG))
print("Device:", DEVICE)
print("Run directory:", RUN_DIR.resolve())

## 2. Reproducibility and data-index construction

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def worker_init_fn(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def _read_json_records(path: Path) -> pd.DataFrame:
    try:
        return pd.read_json(path)
    except ValueError:
        return pd.read_json(path, lines=True)


IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}


def scan_genimage_folders(data_root: str) -> pd.DataFrame:
    # Build an index from <generator>/<train|val>/<ai|nature>/<image>.
    root = Path(data_root)
    if not root.is_dir():
        raise FileNotFoundError(f"Data root does not exist or is not a directory: {root}")

    generator_dirs = sorted(
        path for path in root.iterdir()
        if path.is_dir() and not path.name.startswith("_")
    )
    if not generator_dirs:
        raise FileNotFoundError(f"No generator directories found in {root}")

    rows = []
    for generator_dir in generator_dirs:
        generator = generator_dir.name
        for split in ("train", "val"):
            for label_name in ("ai", "nature"):
                class_dir = generator_dir / split / label_name
                if not class_dir.is_dir():
                    print(f"Warning: missing directory {class_dir}")
                    continue
                count_before = len(rows)
                for path in class_dir.rglob("*"):
                    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                        rows.append({
                            "path": path.relative_to(root).as_posix(),
                            "generator": generator,
                            "split": split,
                            "label": label_name,
                        })
                print(
                    f"Indexed {len(rows) - count_before:,} files from "
                    f"{generator}/{split}/{label_name}"
                )
    if not rows:
        raise FileNotFoundError(
            f"No supported images found under {root}. Expected generator/train|val/ai|nature."
        )
    return pd.DataFrame(rows)


def load_and_validate_data_index(data_root: str, manifest_json: Optional[str] = None) -> pd.DataFrame:
    if manifest_json:
        print(f"Loading combined manifest: {manifest_json}")
        df = _read_json_records(Path(manifest_json))
    else:
        print("No combined manifest supplied; scanning image folders.")
        df = scan_genimage_folders(data_root)

    required = {"path", "label", "generator", "split"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Manifest is missing columns: {sorted(missing)}")

    df = df.copy()
    raw_labels = df["label"].astype(str).str.lower().str.strip()
    label_map = {"nature": 0, "ai": 1, "0": 0, "1": 1}
    unknown = sorted(set(raw_labels) - set(label_map))
    if unknown:
        raise ValueError(f"Unknown labels {unknown}; expected 'nature' or 'ai'.")
    df["label_name"] = raw_labels
    df["label"] = raw_labels.map(label_map).astype(int)
    df["generator"] = df["generator"].astype(str)
    df["split"] = df["split"].astype(str).str.lower()

    if not set(df["label"].unique()).issubset({0, 1}):
        raise ValueError("Labels must be 0=real and 1=fake.")
    if not set(df["split"].unique()).issubset({"train", "val"}):
        raise ValueError("GenImage Compact split must contain only train and val.")
    if df["path"].duplicated().any():
        duplicates = int(df["path"].duplicated().sum())
        raise ValueError(f"Manifest contains {duplicates} duplicate paths.")

    fake_generators = sorted(df.loc[df.label == 1, "generator"].unique())
    print(f"Rows: {len(df):,}")
    print("Fake generators:", fake_generators)
    print(pd.crosstab(df["split"], df["label"], margins=True))
    return df


seed_everything(CFG.seed)
manifest = load_and_validate_data_index(CFG.data_root, CFG.manifest_json)

# Save an exact configuration and a manifest fingerprint with every run.
manifest_hash = hashlib.sha256(
    pd.util.hash_pandas_object(
        manifest[["path", "label", "generator", "split"]].sort_values("path"), index=False
    ).values.tobytes()
).hexdigest()
run_metadata = {**asdict(CFG), "manifest_sha256": manifest_hash, "device": str(DEVICE)}
(RUN_DIR / "config.json").write_text(json.dumps(run_metadata, indent=2))

## 3. Select fixed-budget training and validation rows

For fair comparisons, the **total** fake budget stays constant. For a run with (k) generators, each contributes approximately `train_fake_budget / k` images. Real-image budgets also stay fixed.

In [ ]:
def sample_up_to(frame: pd.DataFrame, n: Optional[int], seed: int) -> pd.DataFrame:
    if n is None or len(frame) <= n:
        return frame.copy()
    return frame.sample(n=n, random_state=seed)


def balanced_label_sample(
    frame: pd.DataFrame,
    generators: Sequence[str],
    label: int,
    total_budget: Optional[int],
    seed: int,
) -> pd.DataFrame:
    frame = frame[(frame.label == label) & frame.generator.isin(generators)].copy()
    missing = sorted(set(generators) - set(frame.generator.unique()))
    if missing:
        class_name = "ai" if label == 1 else "nature"
        raise ValueError(f"No {class_name} rows found for generators: {missing}")
    if total_budget is None:
        return frame

    base, remainder = divmod(total_budget, len(generators))
    pieces = []
    for i, generator in enumerate(sorted(generators)):
        requested = base + int(i < remainder)
        available = frame[frame.generator == generator]
        if len(available) < requested:
            raise ValueError(
                f"{generator} has {len(available):,} rows, fewer than requested {requested:,}. "
                "Lower the budget so all experiments remain balanced."
            )
        pieces.append(available.sample(n=requested, random_state=seed + i))
    return pd.concat(pieces, ignore_index=True)


def fixed_budget_frame(
    frame: pd.DataFrame,
    generators: Sequence[str],
    fake_budget: Optional[int],
    real_budget: Optional[int],
    seed: int,
) -> pd.DataFrame:
    fake = balanced_label_sample(frame, generators, 1, fake_budget, seed)
    real = balanced_label_sample(frame, generators, 0, real_budget, seed + 10_000)
    selected = pd.concat([real, fake], ignore_index=True)
    return selected.sample(frac=1, random_state=seed).reset_index(drop=True)


# Use only selected generators, then split each generator/class stratum so the
# internal validation set represents every selected source and both labels.
official_train = manifest[
    (manifest.split == "train") & manifest.generator.isin(CFG.train_generators)
].copy()
if official_train.empty:
    raise ValueError(f"No official training rows found for {CFG.train_generators}")

train_parts, internal_val_parts = [], []
for (_, _), group in official_train.groupby(["generator", "label"], sort=True):
    if len(group) < 2:
        raise ValueError("Every generator/label training stratum needs at least two images.")
    group_train, group_val = train_test_split(
        group,
        test_size=CFG.internal_val_fraction,
        random_state=CFG.seed,
        shuffle=True,
    )
    train_parts.append(group_train)
    internal_val_parts.append(group_val)

train_pool = pd.concat(train_parts, ignore_index=True)
internal_val_pool = pd.concat(internal_val_parts, ignore_index=True)

train_df = fixed_budget_frame(
    train_pool, CFG.train_generators,
    CFG.train_fake_budget, CFG.train_real_budget, CFG.seed,
)
val_df = fixed_budget_frame(
    internal_val_pool, CFG.train_generators,
    CFG.val_fake_budget, CFG.val_real_budget, CFG.seed,
)

if CFG.debug_max_rows_per_split:
    train_df = sample_up_to(train_df, CFG.debug_max_rows_per_split, CFG.seed)
    val_df = sample_up_to(val_df, CFG.debug_max_rows_per_split, CFG.seed + 1)

print("Train:\n", pd.crosstab(train_df.generator, train_df.label, margins=True))
print("Validation:\n", pd.crosstab(val_df.generator, val_df.label, margins=True))

assert set(train_df.loc[train_df.label == 1, "generator"]).issubset(CFG.train_generators)
assert set(val_df.loc[val_df.label == 1, "generator"]).issubset(CFG.train_generators)

## 4. Dataset, transformations, and data loaders

In [ ]:
weights = ResNet50_Weights.IMAGENET1K_V2
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(CFG.image_size, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(CFG.image_size),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class ManifestImageDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, data_root: str, transform=None):
        self.frame = frame.reset_index(drop=True).copy()
        self.data_root = Path(data_root)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def resolve_path(self, raw_path: str) -> Path:
        path = Path(raw_path)
        return path if path.is_absolute() else self.data_root / path

    def __getitem__(self, idx: int):
        row = self.frame.iloc[idx]
        path = self.resolve_path(str(row["path"]))
        try:
            with Image.open(path) as image:
                image = image.convert("RGB")
        except Exception as exc:
            raise RuntimeError(f"Failed to read image: {path}") from exc
        if self.transform:
            image = self.transform(image)
        return {
            "image": image,
            "label": torch.tensor(float(row["label"]), dtype=torch.float32),
            "path": str(row["path"]),
            "generator": str(row["generator"]),
        }


def make_loader(frame, transform, shuffle=False, batch_size=None):
    generator = torch.Generator()
    generator.manual_seed(CFG.seed)
    return DataLoader(
        ManifestImageDataset(frame, CFG.data_root, transform),
        batch_size=batch_size or CFG.batch_size,
        shuffle=shuffle,
        num_workers=CFG.num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=CFG.num_workers > 0,
        worker_init_fn=worker_init_fn,
        generator=generator,
    )


train_loader = make_loader(train_df, train_transform, shuffle=True)
val_loader = make_loader(val_df, eval_transform)

batch = next(iter(train_loader))
print("Image batch:", batch["image"].shape)
print("Labels:", batch["label"][:10].tolist())
print("Generators:", batch["generator"][:5])

## 5. Model, metrics, and one-epoch functions

In [ ]:
def build_model(pretrained: bool = True) -> nn.Module:
    model = resnet50(weights=weights if pretrained else None)
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model


def safe_auroc(y_true, y_prob) -> float:
    return float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) == 2 else float("nan")


def binary_metrics(y_true, y_prob, threshold=0.5) -> dict:
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "n": int(len(y_true)),
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "auroc": safe_auroc(y_true, y_prob),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    all_labels, all_probs = [], []

    for batch in tqdm(loader, leave=False):
        images = batch["image"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)

        amp_enabled = CFG.use_amp and DEVICE.type == "cuda"
        with torch.set_grad_enabled(training), autocast(enabled=amp_enabled):
            logits = model(images).squeeze(1)
            loss = criterion(logits, labels)

        if training:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        probs = torch.sigmoid(logits)
        total_loss += loss.item() * labels.size(0)
        all_labels.extend(labels.detach().cpu().numpy())
        all_probs.extend(probs.detach().cpu().numpy())

    metrics = binary_metrics(all_labels, all_probs, CFG.decision_threshold)
    metrics["loss"] = total_loss / len(loader.dataset)
    return metrics

## 6. Train and save the best checkpoint

The best epoch is selected using internal-validation AUROC. Official `val` data is never used for model or threshold selection.

In [ ]:
def save_checkpoint(path, model, optimizer, scheduler, epoch, best_val_auroc):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_auroc": best_val_auroc,
        "config": asdict(CFG),
        "manifest_sha256": manifest_hash,
    }, path)


model = build_model(CFG.pretrained).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=CFG.learning_rate, weight_decay=CFG.weight_decay
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs)
scaler = GradScaler(enabled=CFG.use_amp and DEVICE.type == "cuda")

history = []
best_val_auroc = -np.inf
epochs_without_improvement = 0

for epoch in range(1, CFG.epochs + 1):
    start = time.time()
    train_metrics = run_epoch(model, train_loader, criterion, optimizer, scaler)
    val_metrics = run_epoch(model, val_loader, criterion)
    scheduler.step()

    row = {
        "epoch": epoch,
        "learning_rate": optimizer.param_groups[0]["lr"],
        "seconds": time.time() - start,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    history.append(row)
    pd.DataFrame(history).to_csv(RUN_DIR / "history.csv", index=False)
    save_checkpoint(RUN_DIR / "last.pt", model, optimizer, scheduler, epoch, best_val_auroc)

    print(
        f"Epoch {epoch:02d} | train loss {train_metrics['loss']:.4f} | "
        f"val loss {val_metrics['loss']:.4f} | val F1 {val_metrics['f1']:.4f} | "
        f"val AUROC {val_metrics['auroc']:.4f}"
    )

    score = val_metrics["auroc"]
    if np.isfinite(score) and score > best_val_auroc:
        best_val_auroc = score
        epochs_without_improvement = 0
        save_checkpoint(RUN_DIR / "best.pt", model, optimizer, scheduler, epoch, best_val_auroc)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= CFG.early_stopping_patience:
            print("Early stopping.")
            break

print("Best validation AUROC:", best_val_auroc)

## 7. Training curves

In [ ]:
history_df = pd.read_csv(RUN_DIR / "history.csv")
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history_df.epoch, history_df.train_loss, label="train")
axes[0].plot(history_df.epoch, history_df.val_loss, label="val")
axes[0].set(title="Loss", xlabel="Epoch")
axes[1].plot(history_df.epoch, history_df.train_f1, label="train")
axes[1].plot(history_df.epoch, history_df.val_f1, label="val")
axes[1].set(title="F1", xlabel="Epoch", ylim=(0, 1))
axes[2].plot(history_df.epoch, history_df.train_auroc, label="train")
axes[2].plot(history_df.epoch, history_df.val_auroc, label="val")
axes[2].set(title="AUROC", xlabel="Epoch", ylim=(0, 1))
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / "training_curves.png", dpi=160, bbox_inches="tight")
plt.show()

## 8. Load the best model and optionally select a validation threshold

In [ ]:
def predict_frame(model, frame: pd.DataFrame, transform=eval_transform) -> pd.DataFrame:
    loader = make_loader(frame, transform, shuffle=False)
    model.eval()
    rows = []
    with torch.no_grad():
        for batch in tqdm(loader, leave=False):
            images = batch["image"].to(DEVICE, non_blocking=True)
            with autocast(enabled=CFG.use_amp and DEVICE.type == "cuda"):
                probs = torch.sigmoid(model(images).squeeze(1)).cpu().numpy()
            for path, generator, label, prob in zip(
                batch["path"], batch["generator"], batch["label"].numpy(), probs
            ):
                rows.append({
                    "path": path,
                    "generator": generator,
                    "label": int(label),
                    "probability_fake": float(prob),
                })
    return pd.DataFrame(rows)


def best_f1_threshold(labels, probabilities) -> float:
    candidates = np.linspace(0.01, 0.99, 99)
    scores = [f1_score(labels, np.asarray(probabilities) >= t, zero_division=0) for t in candidates]
    return float(candidates[int(np.argmax(scores))])


checkpoint = torch.load(RUN_DIR / "best.pt", map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("Loaded best checkpoint from epoch", checkpoint["epoch"])

threshold = CFG.decision_threshold
if CFG.optimize_threshold_on_validation:
    val_predictions = predict_frame(model, val_df)
    threshold = best_f1_threshold(val_predictions.label, val_predictions.probability_fake)
    print("Validation-selected threshold:", threshold)
else:
    print("Fixed threshold:", threshold)

## 9. Evaluate seen and unseen generators

Each generator is evaluated with all of its fake test images and the same real test pool. Generator-wise macro averages are preferable to pooling when generator test-set sizes differ.

In [ ]:
def generator_test_frame(df: pd.DataFrame, generator: str) -> pd.DataFrame:
    # The official GenImage Compact validation folder is the final evaluation set.
    selected = df[(df.split == "val") & (df.generator == generator)].copy()
    if selected.empty or selected.label.nunique() < 2:
        raise ValueError(f"Test subset for {generator} must contain both real and fake rows.")
    if CFG.debug_max_rows_per_split:
        # Preserve both labels in debug mode.
        pieces = [
            sample_up_to(selected[selected.label == label], CFG.debug_max_rows_per_split // 2, CFG.seed + label)
            for label in (0, 1)
        ]
        selected = pd.concat(pieces, ignore_index=True)
    return selected.reset_index(drop=True)


all_generators = sorted(manifest.loc[manifest.label == 1, "generator"].unique())
metrics_rows = []
prediction_frames = []
(RUN_DIR / "predictions").mkdir(exist_ok=True)

for test_generator in all_generators:
    test_df = generator_test_frame(manifest, test_generator)
    predictions = predict_frame(model, test_df)
    predictions["prediction"] = (predictions.probability_fake >= threshold).astype(int)
    predictions["test_generator"] = test_generator
    predictions["seen"] = test_generator in CFG.train_generators
    predictions.to_csv(RUN_DIR / "predictions" / f"{test_generator}.csv", index=False)
    prediction_frames.append(predictions)

    metrics = binary_metrics(predictions.label, predictions.probability_fake, threshold)
    metrics_rows.append({
        "run_name": CFG.run_name,
        "seed": CFG.seed,
        "k": len(CFG.train_generators),
        "train_generators": "+".join(sorted(CFG.train_generators)),
        "test_generator": test_generator,
        "seen": test_generator in CFG.train_generators,
        **metrics,
    })

results_df = pd.DataFrame(metrics_rows)
results_df.to_csv(RUN_DIR / "per_generator_metrics.csv", index=False)
display(results_df.sort_values(["seen", "test_generator"], ascending=[False, True]))

## 10. Summarize unseen-generator generalization

In [ ]:
unseen = results_df[~results_df.seen]
summary = {
    "run_name": CFG.run_name,
    "train_generators": CFG.train_generators,
    "k": len(CFG.train_generators),
    "threshold": threshold,
    "best_epoch": int(checkpoint["epoch"]),
    "best_validation_auroc": float(checkpoint["best_val_auroc"]),
    "macro_unseen_accuracy": float(unseen.accuracy.mean()) if len(unseen) else None,
    "macro_unseen_f1": float(unseen.f1.mean()) if len(unseen) else None,
    "macro_unseen_auroc": float(unseen.auroc.mean()) if len(unseen) else None,
    "std_unseen_auroc": float(unseen.auroc.std(ddof=1)) if len(unseen) > 1 else None,
}
(RUN_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

plot_df = results_df.sort_values("auroc", ascending=False)
plt.figure(figsize=(10, 4.5))
sns.barplot(data=plot_df, x="test_generator", y="auroc", hue="seen", dodge=False)
plt.axhline(0.5, color="black", linestyle="--", linewidth=1)
plt.ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.title(f"Per-generator AUROC: {CFG.run_name}")
plt.tight_layout()
plt.savefig(RUN_DIR / "per_generator_auroc.png", dpi=160, bbox_inches="tight")
plt.show()

## 11. Confusion matrix and ROC curve for a selected generator

In [ ]:
SELECTED_GENERATOR = all_generators[0]  # change as needed
selected_predictions = pd.read_csv(RUN_DIR / "predictions" / f"{SELECTED_GENERATOR}.csv")
y_true = selected_predictions.label.to_numpy()
y_prob = selected_predictions.probability_fake.to_numpy()
y_pred = (y_prob >= threshold).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.heatmap(
    confusion_matrix(y_true, y_pred, labels=[0, 1]),
    annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0],
    xticklabels=["real", "fake"], yticklabels=["real", "fake"],
)
axes[0].set(title=f"Confusion matrix: {SELECTED_GENERATOR}", xlabel="Predicted", ylabel="True")

fpr, tpr, _ = roc_curve(y_true, y_prob)
axes[1].plot(fpr, tpr, label=f"AUROC = {roc_auc_score(y_true, y_prob):.3f}")
axes[1].plot([0, 1], [0, 1], "k--")
axes[1].set(title=f"ROC: {SELECTED_GENERATOR}", xlabel="False positive rate", ylabel="True positive rate")
axes[1].legend()
fig.tight_layout()
plt.show()

## 12. Run the next generator combination

To run another experiment, change only these fields in the configuration cell, restart the kernel, and run all cells:

```python
# Single-generator baseline
CFG.train_generators = ["stable_diffusion_v_1_4"]
CFG.run_name = "resnet50_k1_sdv14_seed0"

# Two-generator experiment, same total fake budget
CFG.train_generators = ["stable_diffusion_v_1_4", "BigGAN"]
CFG.run_name = "resnet50_k2_sdv14_biggan_seed0"

# Four-generator experiment, still the same total fake budget
CFG.train_generators = ["stable_diffusion_v_1_4", "BigGAN", "Midjourney", "glide"]
CFG.run_name = "resnet50_k4_mix_seed0"
```

Keep `train_fake_budget`, `train_real_budget`, preprocessing, hyperparameters, and splits constant. When comparing models, concatenate their `per_generator_metrics.csv` files and compare the macro average across generators that were unseen in the corresponding run.

## 13. Optional: consolidate finished runs

In [ ]:
metric_files = sorted(Path(CFG.output_root).glob("*/per_generator_metrics.csv"))
if metric_files:
    consolidated = pd.concat([pd.read_csv(path) for path in metric_files], ignore_index=True)
    consolidated.to_csv(Path(CFG.output_root) / "all_experiment_results.csv", index=False)
    display(consolidated)

    macro = (
        consolidated[~consolidated.seen]
        .groupby(["run_name", "k", "train_generators"], as_index=False)
        .agg(
            unseen_accuracy=("accuracy", "mean"),
            unseen_f1=("f1", "mean"),
            unseen_auroc=("auroc", "mean"),
            unseen_auroc_std=("auroc", "std"),
            n_unseen_generators=("test_generator", "nunique"),
        )
    )
    macro.to_csv(Path(CFG.output_root) / "macro_unseen_results.csv", index=False)
    display(macro)
else:
    print("No completed result files found yet.")

## Submission checklist

- `best.pt` and `last.pt` exist for every run.
- `config.json` records the training generators, budgets, seed, and manifest hash.
- `history.csv` and `training_curves.png` show training behavior.
- Every evaluated generator has a prediction CSV from its official `val` folder.
- `per_generator_metrics.csv` reports accuracy, F1, and AUROC.
- `summary.json` reports macro unseen-generator performance.
- No held-out generator was used for validation, threshold selection, or checkpoint selection.
- The total real and fake training budgets stayed fixed when changing (k).